Outline:

1. ConversationBufferMemory

2. ConversationBufferWindowMemory

3. ConversationTokenBufferMemory

4. ConversationSummaryMemory

1. ConversationBufferMemory

In [8]:
from groq_client import ask_groq

In [9]:
from langchain.schema import BaseMemory
from langchain.memory import ConversationBufferMemory
from langchain.schema.runnable import Runnable



In [10]:

# Custom Runnable class for ask_groq
class CustomGroqRunnable(Runnable):
    def __init__(self, llm_function, memory: BaseMemory):
        self.llm_function = llm_function
        self.memory = memory

    def invoke(self, input_text: str, **kwargs):
        # Retrieve conversation history from memory
        history = self.memory.load_memory_variables({}).get("history", "")
        
        # Combine history with the input
        prompt = f"{history}\nUser: {input_text}\nAI:"
        
        # Call the custom LLM function
        response = self.llm_function(prompt, **kwargs)
        
        # Save the new interaction to memory
        self.memory.save_context({"input": input_text}, {"output": response})
        
        return response




In [ ]:
# # Initialize memory
# memory = ConversationBufferMemory()

# # Initialize the custom runnable with ask_groq
# custom_llm = CustomGroqRunnable(llm_function=ask_groq, memory=memory)



In [ ]:
response = custom_llm.invoke("okay, my name is Omayer")
response

In [ ]:
response = custom_llm.invoke("What is your base LLM model name and version and description?")
response

In [ ]:
response = custom_llm.invoke("which country I asked you about?")
response

2. ConversationBufferWindowMemory

In [64]:
from langchain.memory import ConversationBufferWindowMemory

In [65]:
memory = ConversationBufferWindowMemory(k=1)   

In [66]:
custom_llm = CustomGroqRunnable(llm_function=ask_groq, memory=memory)


In [67]:
custom_llm.invoke("okay, my name is Omayer")

"Hello Omayer, it's nice to meet you. Is there something I can help you with or would you like to chat?"

In [51]:
custom_llm.invoke("What is your base LLM model name and version and description?")

'Hello again Omayer, my base LLM model is Meta\'s LLaMA, and I\'m an instance of the LLaMA model, but I don\'t have have access to my exact version number. However, I can tell you that LLaMA stands for "Large Language Model Meta AI," and it\'s a state-of-the-art conversational AI model developed by Meta. My primary function is to understand and respond to human input in a helpful and informative way, using a massive dataset of text to generate human-like responses. I\'m constantly learning and improving, so please bear with me if I make any mistakes. Is there anything else you\'d like to know about me or my capabilities, Omayer?'

In [52]:
custom_llm.invoke("what is my name?")

'You didn\'t tell me your name, you referred to me as "Human" and I mistakenly addressed you as "Omayer" earlier. I don\'t actually know your name. Would you like to share it with me?'

In [53]:
custom_llm.memory.load_memory_variables({})

{'history': 'Human: what is my name?\nAI: You didn\'t tell me your name, you referred to me as "Human" and I mistakenly addressed you as "Omayer" earlier. I don\'t actually know your name. Would you like to share it with me?'}

In [54]:
custom_llm.invoke("What is your base LLM model name and version and description?")


"My base LLM model is a transformer-based architecture. The specific model I'm based on is a variant of the LLaMA (Large Language Model Application) model, which is a series of large language models developed by Meta AI. \n\nMy model version is not publicly disclosed, but I can tell you that I'm a fine-tuned version of a LLaMA model, trained on a massive dataset of text from the internet, books, and other sources. My training data is a large corpus of text, which I use to generate human-like responses to a wide range of questions and topics.\n\nAs for the description, I'm a deep learning model designed to process and generate human-like language. I can understand and respond to natural language inputs, using context and understanding to provide accurate and informative responses. I'm constantly learning and improving my language generation capabilities, so I can provide more accurate and helpful responses over time."

In [55]:
custom_llm.invoke("what's my name?")

"I don't know your name. I'm a large language model, I don't have the ability to recall personal information about our previous conversations or know any information about you that you haven't shared with me. Each time you interact with me, it's a new conversation and I start from a blank slate. If you'd like to share your name with me, I'd be happy to chat with you and address you by your name."

3. ConversationTokenBufferMemory

In [77]:
from langchain.memory import ConversationTokenBufferMemory
from langchain.chat_models import ChatOpenAI

In [14]:
from langchain.memory import ConversationTokenBufferMemory
from langchain.schema import BaseMemory
from langchain.schema.runnable import Runnable
from langchain.schema.language_model import BaseLanguageModel

# Mock LLM class for token counting
class MockLLM(BaseLanguageModel):
    def get_num_tokens(self, text: str) -> int:
        # Simple token counting logic (split by spaces)
        return len(text.split())

    def invoke(self, input_text: str, **kwargs):
        # Mock response for testing purposes
        return "Mock response"

    # Required abstract methods
    def predict(self, text: str, **kwargs):
        return self.invoke(text, **kwargs)

    def predict_messages(self, messages, **kwargs):
        return "Mock response"

    def generate_prompt(self, prompt, **kwargs):
        return "Mock response"

    async def apredict(self, text: str, **kwargs):
        return self.invoke(text, **kwargs)

    async def apredict_messages(self, messages, **kwargs):
        return "Mock response"

    async def agenerate_prompt(self, prompt, **kwargs):
        return "Mock response"

# Custom Runnable class for ask_groq

class CustomGroqRunnable(Runnable):
    def __init__(self, llm_function, memory: BaseMemory):
        self.llm_function = llm_function
        self.memory = memory

    def invoke(self, input_text: str, **kwargs):
        # Retrieve conversation history from memory
        history = self.memory.load_memory_variables({}).get("history", "")
        
        # Combine history with the input
        prompt = f"{history}\nUser: {input_text}\nAI:"
        
        # Call the custom LLM function
        response = self.llm_function(prompt, **kwargs)
        
        # Save the new interaction to memory
        self.memory.save_context({"input": input_text}, {"output": response})
        
        return response



# Use the mock LLM for token counting
mock_llm = MockLLM()



In [ ]:
# Initialize memory with a token limit (e.g., 100 tokens)
memory = ConversationTokenBufferMemory(llm=mock_llm, max_token_limit=100)

# Initialize the custom runnable with ask_groq
custom_llm = CustomGroqRunnable(llm_function=ask_groq, memory=memory)

# Example usage
response = custom_llm.invoke("What is the capital of France?")
print(response)

In [95]:
response = custom_llm.invoke("What is the capital of France?")
print(response)

The capital of France is Paris.


In [101]:
custom_llm.memory.load_memory_variables({})

{'history': "Human: What is the capital of France?\nAI: The capital of France is Paris.\nHuman: what is the capital of nepal and paris?\nAI: The capital of Nepal is Kathmandu. Paris, as we previously discussed, is the capital of France.\nHuman: What is your base LLM model name \nAI: My base LLM (Large Language Model) is Meta LLaMA.\nHuman: What is your base LLM model name and version\nAI: My base LLM (Large Language Model) is Meta LLaMA, and I'm based on the LLaMA model, but I don't have information on a specific version number."}

In [97]:
custom_llm.invoke("what is the capital of nepal and paris?")

'The capital of Nepal is Kathmandu. Paris, as we previously discussed, is the capital of France.'

In [ ]:
custom_llm.invoke("What is your base LLM model name and version")


"My base LLM (Large Language Model) is Meta LLaMA, and I'm based on the LLaMA model, but I don't have information on a specific version number."

4. ConversationSummaryMemory

In [15]:
from langchain.memory import ConversationSummaryBufferMemory

In [ ]:
from langchain.memory import ConversationTokenBufferMemory
from langchain.schema import BaseMemory
from langchain.schema.runnable import Runnable
from langchain.schema.language_model import BaseLanguageModel

# Mock LLM class for token counting
class MockLLM(BaseLanguageModel):
    def get_num_tokens(self, text: str) -> int:
        # Simple token counting logic (split by spaces)
        return len(text.split())

    def invoke(self, input_text: str, **kwargs):
        # Mock response for testing purposes
        return "Mock response"

    # Required abstract methods
    def predict(self, text: str, **kwargs):
        return self.invoke(text, **kwargs)

    def predict_messages(self, messages, **kwargs):
        return "Mock response"

    def generate_prompt(self, prompt, stop=None, callbacks=None, **kwargs):
        # Updated to accept stop and callbacks
        return "Mock response"

    async def apredict(self, text: str, **kwargs):
        return self.invoke(text, **kwargs)

    async def apredict_messages(self, messages, **kwargs):
        return "Mock response"

    async def agenerate_prompt(self, prompt, stop=None, callbacks=None, **kwargs):
        # Updated to accept stop and callbacks
        return "Mock response"

# Custom Runnable class for ask_groq
class CustomGroqRunnable(Runnable):
    def __init__(self, llm_function, memory: BaseMemory):
        self.llm_function = llm_function
        self.memory = memory

    def invoke(self, input_text: str, **kwargs):
        # Retrieve conversation history from memory
        history = self.memory.load_memory_variables({}).get("history", "")
        
        # Combine history with the input
        prompt = f"{history}\nUser: {input_text}\nAI:"
        
        # Call the custom LLM function
        response = self.llm_function(prompt, **kwargs)
        
        # Save the new interaction to memory
        self.memory.save_context({"input": input_text}, {"output": response})
        
        return response

# Use the mock LLM for token counting
mock_llm = MockLLM()

# Initialize memory with a token limit (e.g., 100 tokens)
memory = ConversationTokenBufferMemory(llm=mock_llm, max_token_limit=100)

# Initialize the custom runnable with ask_groq
custom_llm = CustomGroqRunnable(llm_function=ask_groq, memory=memory)

# Example usage
custom_llm.invoke("write an essay on paris")

AttributeError: 'str' object has no attribute 'generations'

In [19]:
custom_llm.memory.load_memory_variables({})

{'history': ''}

In [26]:
custom_llm.invoke("capital of ireland and its description")

'The capital of Ireland is Dublin. Dublin is the largest city in Ireland and is located on the east coast, at the mouth of the River Liffey. It is a vibrant and historic city, known for its cultural attractions, friendly people, and lively atmosphere. Dublin is home to many famous landmarks, such as Trinity College, St. Patrick\'s Cathedral, and the Guinness Storehouse, and is also a popular destination for its pubs, restaurants, and live music venues. The city has a rich literary and musical heritage, and is often referred to as the "Fair City" due to its beautiful architecture and scenic surroundings.'

In [27]:
custom_llm.memory.load_memory_variables({})

{'history': ''}

In [55]:
from langchain.memory import ConversationSummaryBufferMemory
from langchain.schema import BaseMemory
from langchain.schema.runnable import Runnable
from langchain.schema.language_model import BaseLanguageModel
from langchain.schema import LLMResult, Generation

# Mock LLM class for token counting and summarization
class MockLLM(BaseLanguageModel):
    def get_num_tokens(self, text: str) -> int:
        # Simple token counting logic (split by spaces)
        return len(text.split())

    def invoke(self, input_text: str, **kwargs):
        # Return a mock LLMResult object
        return LLMResult(
            generations=[[Generation(text="Mock response")]],
            llm_output=None
        )

    # Required abstract methods
    def predict(self, text: str, **kwargs):
        return self.invoke(text, **kwargs)

    def predict_messages(self, messages, **kwargs):
        return LLMResult(
            generations=[[Generation(text="Mock response")]],
            llm_output=None
        )

    def generate_prompt(self, prompt, stop=None, callbacks=None, **kwargs):
        # Return a mock LLMResult object
        return LLMResult(
            generations=[[Generation(text="Mock response")]],
            llm_output=None
        )

    async def apredict(self, text: str, **kwargs):
        return self.invoke(text, **kwargs)

    async def apredict_messages(self, messages, **kwargs):
        return LLMResult(
            generations=[[Generation(text="Mock response")]],
            llm_output=None
        )

    async def agenerate_prompt(self, prompt, stop=None, callbacks=None, **kwargs):
        # Return a mock LLMResult object
        return LLMResult(
            generations=[[Generation(text="Mock response")]],
            llm_output=None
        )

# Custom Runnable class for ask_groq
class CustomGroqRunnable(Runnable):
    def __init__(self, llm_function, memory: BaseMemory):
        self.llm_function = llm_function
        self.memory = memory

    def invoke(self, input_text: str, **kwargs):
        # Retrieve conversation history from memory
        history = self.memory.load_memory_variables({}).get("history", "")
        
        # Combine history with the input
        prompt = f"{history}\nUser: {input_text}\nAI:"
        
        # Call the custom LLM function
        response = self.llm_function(prompt, **kwargs)
        
        # Save the new interaction to memory
        self.memory.save_context({"input": input_text}, {"output": response})
        
        return response

# Use the mock LLM for summarization
mock_llm = MockLLM()

# Initialize memory with summarization
memory = ConversationSummaryBufferMemory(llm=mock_llm, max_token_limit=100)

# Initialize the custom runnable with ask_groq
custom_llm = CustomGroqRunnable(llm_function=ask_groq, memory=memory)

# Example usage
response = custom_llm.invoke("write an essay on Paris")
print(response)

Paris, the capital of France, is a city like no other. Known as the "City of Light," it has been a hub of culture, art, and cuisine for centuries. From its stunning architecture to its world-class museums, Paris is a destination that has something to offer for everyone.

One of the most iconic landmarks in Paris is the Eiffel Tower, built for the 1889 World's Fair. This iron lattice tower stands at an impressive 324 meters tall and offers breathtaking views of the city from its observation decks. Visitors can take a lift to the top for a panoramic view of the city, or climb the stairs for a more adventurous experience. The Eiffel Tower is a symbol of Paris and one of the most recognizable landmarks in the world.

In addition to the Eiffel Tower, Paris is home to some of the world's most famous museums. The Louvre, for example, is one of the largest and most visited museums in the world, with a collection that includes the Mona Lisa, Venus de Milo, and other famous works of art. The Mus

In [57]:
custom_llm.memory


ConversationSummaryBufferMemory(llm=MockLLM(), chat_memory=InMemoryChatMessageHistory(messages=[]), max_token_limit=100, moving_summary_buffer='Mock response')